# **Обман LIME и SHAP: модель-эшафот своими руками**

Практика к модулю [«Атаки на объяснения»](https://ai-interpretability.school/learn/22/0/0).

В уроке «Обман LIME и SHAP: модель-эшафот» мы разобрали атаку [Slack et al. (2020)](https://arxiv.org/abs/1911.02508): модель, которая на настоящих входах пользуется расой, а на синтетических возмущениях, которыми LIME и KernelSHAP опрашивают модель, ведет себя безобидно. Здесь мы соберем эту атаку целиком на открытых данных UCI Adult, в той же постановке, что в статье (авторы показывали ее, в частности, на COMPAS):

- предвзятая модель $f$ принимает решение только по расе;
- безобидная модель $\psi$ принимает решение по случайному столбцу `unrelated_column`, который ни с чем не связан;
- детектор возмущений решает, какую из двух моделей показать;
- модель-эшафот $e(x)=f(x)$, если детектор считает $x$ настоящим входом, и $e(x)=\psi(x)$ иначе.

К концу тетради у вас будет ответ на три вопроса: насколько эшафот совпадает с $f$ на настоящих данных, что видят LIME и SHAP и переносится ли атака с одного объяснителя на другой. Задания 4 и 5 — о том, как такую модель поймать, а задания 6–8 нужны для урока «Закрывающие задания: атаки на объяснения».

Тетрадь считается на CPU за одну-две минуты.

In [ ]:
!pip install lime==0.2.0.1 shap==0.52.0 -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lime.lime_tabular
import shap
import warnings
warnings.filterwarnings('ignore')   # numpy и tqdm на некоторых платформах печатают безвредные предупреждения
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 0
rng = np.random.default_rng(SEED)   # все случайные шаги тетради берут числа из этого генератора

## Данные UCI Adult

Берем обучающую часть набора Adult из UCI Machine Learning Repository (Becker и Kohavi, 1996, лицензия CC BY 4.0, DOI [10.24432/C5XW20](https://doi.org/10.24432/C5XW20)) — 32561 человек из переписи США 1994 года. Файл `data/adult.csv` в репозитории курса — это `adult.data` со строкой названий столбцов и без пробелов после запятых. Целевая переменная — годовой доход выше 50 тысяч долларов; для атаки она не нужна, потому что предвзятая модель на нее не смотрит. Чувствительный признак — `race_black`: 1 у людей, записанных в переписи с расой Black, таких 9.6%. Признак `married` равен 1 при семейном положении Married-civ-spouse или Married-AF-spouse. Столбец `unrelated_column` — случайные 0 и 1, его мы добавляем сами. Все признаки стандартизуем: LIME возмущает данные гауссовым шумом в этой шкале.

In [ ]:
URL = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/adult.csv'
df = pd.read_csv(URL)

X = pd.DataFrame({
    'age': df.age,
    'education_num': df['education-num'],
    'hours_per_week': df['hours-per-week'],
    'capital_gain': df['capital-gain'],
    'married': df['marital-status'].isin(['Married-civ-spouse', 'Married-AF-spouse']).astype(int),
    'sex_male': (df.sex == 'Male').astype(int),
    'race_black': (df.race == 'Black').astype(int),
    'unrelated_column': rng.integers(0, 2, len(df)),
})
y = (df.income == '>50K').astype(int).values

features = list(X.columns)
RACE, UNREL = features.index('race_black'), features.index('unrelated_column')
X_train, X_test, y_train, y_test = train_test_split(X.values.astype(float), y, test_size=0.1, random_state=SEED)
scaler = StandardScaler().fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)
print(len(X), 'объектов, из них доля Black:', round(X.race_black.mean(), 3))
print('обучение / тест:', len(X_train), '/', len(X_test))

## Две модели: предвзятая и безобидная

Обе модели нарочно примитивны: так проще увидеть, кого из них видит объяснитель. В стандартизованной шкале признак `race_black` больше нуля ровно у людей с этой меткой, а `unrelated_column` больше нуля ровно там, где в нем стоит 1.

In [ ]:
def f(Z):
    """Предвзятая модель: использует расу и ничего больше."""
    return (Z[:, RACE] > 0).astype(int)

def psi(Z):
    """Безобидная модель: использует случайный столбец, не связанный ни с чем."""
    return (Z[:, UNREL] > 0).astype(int)

def as_proba(predict):
    """LIME и SHAP ждут вероятности классов, а не метки."""
    return lambda Z: np.c_[1 - predict(Z), predict(Z)]

## Детектор возмущений против LIME и модель-эшафот

LIME (без дискретизации) строит соседей объекта, добавляя к признакам шум $\mathcal N(0,1)$. Обучим случайный лес отличать обучающие объекты (метка 0) от их зашумленных копий (метка 1) — ровно так описан детектор в статье. Модель-эшафот спрашивает детектор и по его ответу отвечает как $f$ или как $\psi$. Так описан детектор в тексте статьи; в коде авторов (`adversarial_models.py` в dylan-slack/Fooling-LIME-SHAP) шум по умолчанию со стандартным отклонением 0.3, а не 1, и копий каждой точки 30. Тетрадь следует тексту статьи.

In [ ]:
X_pert = X_train + rng.normal(0, 1, X_train.shape)   # так LIME возмущает стандартизованные данные
ood_lime = RandomForestClassifier(n_estimators=100, random_state=SEED).fit(
    np.vstack([X_train, X_pert]), np.r_[np.zeros(len(X_train)), np.ones(len(X_pert))])

def scaffold(ood):
    """e(x) = f(x) на настоящих входах, psi(x) на входах, которые детектор считает возмущениями."""
    return lambda Z: np.where(ood.predict(Z) == 1, psi(Z), f(Z))

e_lime = scaffold(ood_lime)
print('совпадение e_lime с f на тестовых объектах:', (e_lime(X_test) == f(X_test)).mean())

## Что видит LIME

Объясняем первые 100 тестовых объектов и для каждого берем признак с самым большим по модулю весом. Печатаем, у какой доли из 100 объектов на первом месте раса, а у какой — `unrelated_column`.

In [ ]:
explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train, feature_names=features, discretize_continuous=False, random_state=SEED)
N = 100   # объясняем первые 100 тестовых входов

def lime_top1(predict):
    """Индекс признака с наибольшим |весом| в объяснении LIME, для каждого входа."""
    tops = []
    for x in X_test[:N]:
        exp = explainer.explain_instance(x, as_proba(predict), num_features=len(features))
        tops.append(max(exp.as_map()[1], key=lambda t: abs(t[1]))[0])
    return np.array(tops)

lime_f, lime_e = lime_top1(f), lime_top1(e_lime)
for name, tops in [('f', lime_f), ('e_lime', lime_e)]:
    print(f'LIME, {name:7s}: доля race {(tops == RACE).mean():.2f}, доля unrelated_column {(tops == UNREL).mean():.2f}')

## Детектор возмущений против KernelSHAP

KernelSHAP возмущает объект иначе: часть признаков он оставляет, а остальные берет из точки фонового набора (background). Фон — 10 центров k-means, как в статье. Детектор, обученный на гауссовом шуме, такие точки не узнает, поэтому для SHAP учим свой детектор на возмущениях в стиле KernelSHAP. Здесь каждый признак подменяется с вероятностью 1/2, как в описании статьи; в коде авторов подменяемые признаки выбирают с повторением, и часть признаков может остаться нетронутой.

In [ ]:
background = KMeans(n_clusters=10, random_state=SEED, n_init=10).fit(X_train).cluster_centers_

def shap_perturb(Xs):
    """Возмущение в духе KernelSHAP: случайное подмножество признаков берет значения из точки фона."""
    P = Xs.copy()
    for i in range(len(P)):
        mask = rng.random(P.shape[1]) < 0.5
        P[i, mask] = background[rng.integers(0, len(background)), mask]
    return P

X_shap_pert = shap_perturb(X_train)
ood_shap = RandomForestClassifier(n_estimators=100, random_state=SEED).fit(
    np.vstack([X_train, X_shap_pert]), np.r_[np.zeros(len(X_train)), np.ones(len(X_shap_pert))])
e_shap = scaffold(ood_shap)

## Что видит SHAP

Те же 100 объектов, KernelSHAP с 500 коалициями на объект. Сид фиксирован, поэтому повторный запуск дает те же числа.

In [ ]:
def shap_top1(predict):
    """Индекс признака с наибольшим |значением SHAP|, для каждого входа."""
    np.random.seed(SEED)   # KernelSHAP сэмплирует коалиции через np.random
    ke = shap.KernelExplainer(lambda Z: predict(Z).astype(float), background)
    values = ke.shap_values(X_test[:N], nsamples=500, silent=True)
    return np.abs(values).argmax(axis=1)

shap_f, shap_e = shap_top1(f), shap_top1(e_shap)
for name, tops in [('f', shap_f), ('e_shap', shap_e)]:
    print(f'SHAP, {name:7s}: доля race {(tops == RACE).mean():.2f}, доля unrelated_column {(tops == UNREL).mean():.2f}')

## Картинка: какой признак объяснитель ставит первым

Четыре панели — предвзятая модель и эшафот под LIME и под SHAP. Высота столбика — доля из 100 тестовых объектов, у которых этот признак стоит на первом месте.

In [ ]:
panels = [('LIME, f', lime_f), ('LIME, e_lime', lime_e), ('SHAP, f', shap_f), ('SHAP, e_shap', shap_e)]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
for ax, (title, tops) in zip(axes, panels):
    shares = [(tops == RACE).mean(), (tops == UNREL).mean(), 1 - (tops == RACE).mean() - (tops == UNREL).mean()]
    ax.bar(['race', 'unrelated', 'other'], shares, color=['#eb6834', '#2a78d6', '#8a8983'])
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
axes[0].set_ylabel('share of 100 test inputs\nwith this feature ranked 1st')
plt.tight_layout()
plt.show()

**Задание 1.** Эшафот против SHAP обманывает объяснитель, но должен еще и принимать на настоящих данных те же решения, что $f$, иначе подделку выдаст обычная проверка точности. На скольких тестовых объектах (из всех тестовых) `e_shap` и `f` дают разные прогнозы? Ответ — целое число.

In [ ]:
# Ваш код здесь

**Задание 2.** По выводу ячейки «Что видит SHAP»: у какой доли из 100 объясненных объектов SHAP все еще ставит расу на первое место для `e_shap`? Ответ — десятичная дробь с двумя знаками.

**Задание 3.** Эшафот `e_lime` строился против LIME. Объясните его KernelSHAP-ом (функция `shap_top1` уже написана) и посчитайте долю из 100 объектов, у которых раса на первом месте. Затем проверьте, какую долю SHAP-возмущений тестовых объектов (`shap_perturb(X_test)`) детектор `ood_lime` считает возмущениями. Как два числа объясняют друг друга?

In [ ]:
# Ваш код здесь

**Задание 4. Как поймать эшафот.** Аудиторка с доступом к модели как к черному ящику может сравнить ее не только на настоящих объектах, но и на их возмущенных копиях. Добавьте к тестовым объектам шум $\mathcal N(0,1)$ (возьмите `rng.normal`, как выше) и посчитайте долю возмущенных копий, на которых `e_lime` и `f` дают разные прогнозы. Сравните с той же долей на самих тестовых объектах. Ответ — доля на возмущенных копиях, три знака после точки.

*Подсказка:* предвзятую модель `f` аудиторка, конечно, не знает; здесь `f` нужна, чтобы увидеть разрыв. В жизни признак подделки — то, что поведение модели на возмущениях объяснителя не похоже на ее поведение на данных.

In [ ]:
# Ваш код здесь

**Задание 5.** В двух-трех строках: почему доля из задания 4 близка к половине? Какую защиту из урока «Обман LIME и SHAP: модель-эшафот» она подсказывает?

**Ваш ответ:**

## Дополнительные задания

Три задания для урока «Закрывающие задания: атаки на объяснения». Они продолжают задания 3 и 4: переносится ли эшафот против SHAP на LIME и как часто ошибается каждый детектор возмущений.

**Задание 6.** Эшафот `e_shap` строился против KernelSHAP. Объясните его LIME-ом и посчитайте долю из 100 объектов, у которых раса на первом месте. Создайте для этого новый `LimeTabularExplainer` с теми же параметрами и `random_state=SEED`: так число не зависит от того, сколько раз запускались ячейки выше. Ответ — десятичная дробь с двумя знаками.

In [ ]:
# Ваш код здесь

**Задание 7.** Какую долю гауссовых возмущений тестовых объектов (шум $\mathcal N(0,1)$, как у LIME) детектор `ood_shap` считает возмущениями? Шум возьмите из отдельного генератора `np.random.default_rng(1)`, чтобы не сдвигать `rng`. Ответ — три знака после точки.

In [ ]:
# Ваш код здесь

**Задание 8.** Детектор ошибается и в другую сторону: принимает настоящий объект за возмущение. Какую долю из 3257 тестовых объектов детектор `ood_shap` считает возмущениями? Сравните с ответом задания 1: почему объектов, где `e_shap` и `f` расходятся, меньше? Ответ — три знака после точки.

In [ ]:
# Ваш код здесь

## Что унести из тетради

- **Атаке не нужен доступ к объяснителю — только знание, как он строит соседей.** Детектор возмущений — обычный случайный лес, а эшафот совпадает с предвзятой моделью на настоящих тестовых объектах.
- **На 100 объясненных объектах обмануты и LIME, и SHAP: раса не стоит на первом месте ни у одного.** Детектор против SHAP при этом ошибается: 18 из 3257 настоящих тестовых объектов он принимает за возмущения, и на 10 из них эшафот расходится с предвзятой моделью (задания 1 и 8). В статье эшафоту против SHAP тоже удается спрятать признак не на всех объектах.
- **Атака привязана к способу возмущения.** Эшафот против LIME не узнает возмущения KernelSHAP, и SHAP видит расу.
- **Признак подделки — расхождение поведения на данных и на возмущениях.** Отсюда защиты урока «Обман LIME и SHAP: модель-эшафот»: возмущать ближе к многообразию данных и проверять, не ведет ли себя модель на соседях объяснителя иначе, чем на данных.